# 3.3 Code Brief: Tuning Tree-Based Models

## Key Concepts

- Same tuning API for all three models: `GridSearchCV`/`RandomizedSearchCV` with `scoring='f1'`.
- Decision Tree: small search space -> `GridSearchCV`. Random Forest / XGBoost: large search space -> `RandomizedSearchCV`.
- XGBoost early stopping reuses `xgb_search.best_params_` (minus `n_estimators`) rather than guessing new hyperparameters.
- Impute with TRAIN medians only, never test's own (avoids leakage).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
course3_models = '/models/'

In [ ]:
import numpy as np
import pandas as pd
import joblib, os
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                      StratifiedKFold, cross_val_score)
from sklearn.metrics import make_scorer, f1_score
import time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ARTIFACT_DIR = f'{project_path}{course3_models}'   # artifacts persist here
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Load and prepare data (same as 3.2)
train_df = pd.read_csv(f'{project_path}{data_filepath}training.csv')
test_df = pd.read_csv(f'{project_path}{data_filepath}testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features],
                           columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features],
                          columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

# --- Capture medians BEFORE filling so they persist; apply TRAIN medians to both ---
train_medians = train_enc.median()
train_enc = train_enc.fillna(train_medians)
test_enc  = test_enc.fillna(train_medians)

X_train, y_train = train_enc, train_df['DEPARTED']
X_test, y_test = test_enc, test_df['DEPARTED']

feature_columns = list(train_enc.columns)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Data loaded: {X_train.shape[0]:,} training, {X_test.shape[0]:,} testing samples")

# --- Persist artifacts for 3.4 ---
joblib.dump(feature_columns, f'{ARTIFACT_DIR}feature_columns.pkl')
joblib.dump(train_medians,   f'{ARTIFACT_DIR}train_medians.pkl')
print(f"Saved {len(feature_columns)} feature columns + {len(train_medians)} medians → {ARTIFACT_DIR}")

## Tuning Decision Trees

In [ ]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
param_grid = {
    'max_depth': [3, 5, 8, 12],
    'min_samples_split': [10, 20, 50],
    'min_samples_leaf': [5, 10, 20],
    'class_weight': ['balanced', None]
}
dt_search = GridSearchCV(dt, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
dt_search.fit(X_train, y_train)

dt_best = dt_search.best_estimator_
dt_prob = dt_best.predict_proba(X_test)[:, 1]
dt_pred = (dt_prob >= 0.5).astype(int)

print(f"Decision Tree Best CV F1: {dt_search.best_score_:.4f}")
print(f"Decision Tree Best Parameters: {dt_search.best_params_}")
print(f"Decision Tree Test F1: {f1_score(y_test, dt_pred):.4f}")

## Tuning Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=RANDOM_STATE)
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}
rf_search = RandomizedSearchCV(rf, param_distributions, n_iter=30, cv=cv, scoring='f1', n_jobs=-1, verbose=1, random_state=RANDOM_STATE)
rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
rf_prob = rf_best.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)

print(f"Random Forest Best CV F1: {rf_search.best_score_:.4f}")
print(f"Random Forest Best Parameters: {rf_search.best_params_}")
print(f"Random Forest Test F1: {f1_score(y_test, rf_pred):.4f}")

## Tuning XGBoost

In [ ]:
neg_count = y_train.value_counts()[0]
pos_count = y_train.value_counts()[1]
scale_pos_weight_value = neg_count / pos_count

xgb = XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')
param_distributions = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'scale_pos_weight': [scale_pos_weight_value]
}
xgb_search = RandomizedSearchCV(xgb, param_distributions, n_iter=30, cv=cv, scoring='f1', n_jobs=-1, verbose=1, random_state=RANDOM_STATE)
xgb_search.fit(X_train, y_train)

xgb_best = xgb_search.best_estimator_
xgb_prob = xgb_best.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_prob >= 0.5).astype(int)

print(f"XGBoost Best CV F1: {xgb_search.best_score_:.4f}")
print(f"XGBoost Best Parameters: {xgb_search.best_params_}")
print(f"XGBoost Test F1: {f1_score(y_test, xgb_pred):.4f}")

## XGBoost Early Stopping

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Split training data 80/20 for early stopping validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=RANDOM_STATE, stratify=y_train
)

# 2. Get best params from search (excluding n_estimators to let early stopping work)
best_params = xgb_search.best_params_.copy()
best_params.pop('n_estimators', None)

# 3. Build XGBoost with best_params and early_stopping_rounds=20
# Set n_estimators (max rounds) to 1000
xgb_early = XGBClassifier(
    **best_params,
    n_estimators=1000,
    early_stopping_rounds=20,
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric='logloss'
)

# 4. Fit with eval_set on the validation split
xgb_early.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# 5. Evaluate and display results
best_iteration = xgb_early.best_iteration
xgb_early_pred = xgb_early.predict(X_test)
test_f1 = f1_score(y_test, xgb_early_pred)

print(f"Best Iteration: {best_iteration}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"\nEarly stopping stopped training at round {best_iteration + 1}.")
print(f"It prevented {1000 - (best_iteration + 1)} unnecessary boosting rounds.")

## Final Comparison: Tuned Models

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Define the models to compare
# dt_best and rf_best were extracted from the search objects earlier
# xgb_early is the model from the early stopping section
comparison_models = {
    'Decision Tree': dt_best,
    'Random Forest': rf_best,
    'XGBoost (Early Stopping)': xgb_early
}

comparison_results = []

for name, model in comparison_models.items():
    # Generate predictions on the test set
    y_pred = model.predict(X_test)

    # Calculate metrics
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    comparison_results.append({
        'Model': name,
        'F1 Score': f1,
        'Precision': prec,
        'Recall': rec
    })

# Create a DataFrame for display
comparison_df = pd.DataFrame(comparison_results).set_index('Model')

# Sort by F1 Score for easier comparison
comparison_df = comparison_df.sort_values(by='F1 Score', ascending=False)

print("Final Comparison of Tuned Models on Test Set:")
display(comparison_df.style.format('{:.4f}'))

## Save the Tuned Models

In [ ]:
import joblib

# Define the directory where models will be saved
model_save_path = f'{project_path}{course3_models}'

# Save the Decision Tree model
joblib.dump(dt_best, f'{model_save_path}dt_tuned_f1.pkl')
print(f"Decision Tree model saved to: {model_save_path}dt_tuned_f1.pkl")

# Save the Random Forest model
joblib.dump(rf_best, f'{model_save_path}rf_tuned_f1.pkl')
print(f"Random Forest model saved to: {model_save_path}rf_tuned_f1.pkl")

# Save the XGBoost model (with early stopping)
joblib.dump(xgb_early, f'{model_save_path}xgb_tuned_f1.pkl')
print(f"XGBoost (Early Stopping) model saved to: {model_save_path}xgb_tuned_f1.pkl")

## Tuning Cheat Sheet

| Model | Most Impactful Parameters | Tuning Strategy |
|:------|:------------------------|:----------------|
| **Decision Tree** | `max_depth`, `min_samples_leaf` | GridSearchCV (small space) |
| **Random Forest** | `n_estimators`, `max_depth`, `max_features` | RandomizedSearchCV |
| **XGBoost** | `learning_rate`, `max_depth`, `n_estimators` | RandomizedSearchCV + early stopping |

**Next:** `3.4 Evaluating Tree-Based Models`